## Initializing the NUTS-HMC sampler

In this example, we show how to use parameter estimates returned by any of Stan's inference algorithms as initial values for Stan's NUTS-HMC sampler. These include:

* [Pathfinder ](https://mc-stan.org/docs/cmdstan-guide/pathfinder-config.html) 
* [ADVI ](https://mc-stan.org/docs/cmdstan-guide/variational_config.html) 
* [Laplace](https://mc-stan.org/docs/cmdstan-guide/laplace_sample_config.html)
* [Optimization](https://mc-stan.org/docs/cmdstan-guide/optimize_config.html)
* [NUTS-HMC MCMC](https://mc-stan.org/docs/cmdstan-guide/mcmc_config.html)

By default, the NUTS-HMC sampler randomly initializes all (unconstrained) model parameters uniformly in the interval (-2, 2).  If this interval is far from the typical set of the posterior, initializing sampling from these approximation algorithms can speed up and improve adaptation.

### Model and data

The Stan model and data are taken from the [posteriordb package](https://github.com/stan-dev/posteriordb).

We use the [blr model](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/models/stan/blr.stan),
a Bayesian standard linear regression model with noninformative priors,
and its corresponding simulated dataset [sblri.json](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/data/data/sblri.json.zip),
which was simulated via script [sblr.R](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/data/data-raw/sblr/sblr.R).
For convenience, this example assumes the posteriordb model and data are local, in files `blr.stan` and `sblri.json`.

In [1]:
import os
from cmdstanpy import CmdStanModel

stan_file = 'blr.stan' # basic linear regression
data_file = 'sblri.json' # simulated data

model = CmdStanModel(stan_file=stan_file)

print(model.code())

16:29:21 - cmdstanpy - INFO - compiling stan file /home/runner/work/cmdstanpy/cmdstanpy/docsrc/users-guide/examples/blr.stan to exe file /home/runner/work/cmdstanpy/cmdstanpy/docsrc/users-guide/examples/blr


16:29:31 - cmdstanpy - INFO - compiled model executable: /home/runner/work/cmdstanpy/cmdstanpy/docsrc/users-guide/examples/blr


data {
  int<lower=0> N;
  int<lower=0> D;
  matrix[N, D] X;
  vector[N] y;
}
parameters {
  vector[D] beta;
  real<lower=0> sigma;
}
model {
  // prior
  target += normal_lpdf(beta | 0, 10);
  target += normal_lpdf(sigma | 0, 10);
  // likelihood
  target += normal_lpdf(y | X * beta, sigma);
}




### Demonstration with Stan's `pathfinder` method

Initializing the sampler with estimates from any previous inference algorithm follows the same general usage pattern. First, we call the 
corresponding method on the `CmdStanModel` object. From the resulting fit, we call the `.create_inits()` 
method to construct a set of per-chain initializations for the model parameters. To make it explicit, 
we will walk through the process using the `pathfinder` method (which wraps the 
CmdStan [pathfinder ](https://mc-stan.org/docs/cmdstan-guide/pathfinder-config.html) method).

Pathfinder locates normal approximations to the target
density along a quasi-Newton optimization path, with local covariance
estimated using the negative inverse Hessian estimates produced by the
LBFGS optimizer. Pathfinder returns draws from the Gaussian approximation
with the lowest estimated Kullback-Leibler (KL) divergence to the true
posterior.
By default, CmdStanPy runs multi-path Pathfinder which returns an importance-resampled 
set of draws over the outputs of 4 independent single-path Pathfinders.
This better matches non-normal target densities and also mitigates
the problem of L-BFGS getting stuck at local optima or in saddle points on plateaus.

We obtain Pathfinder estimates by calling the `.pathfinder()` method which returns a `CmdStanPathfinder` object:

In [2]:
pathfinder_fit = model.pathfinder(data=data_file, seed=123)

16:29:31 - cmdstanpy - INFO - Chain [1] start processing


16:29:31 - cmdstanpy - INFO - Chain [1] done processing


Posteriordb provides reference posteriors for all models. For the blr model, conditioned on the dataset `sblri.json`, the reference posteriors can be found in the [sblri-blr.json](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/reference_posteriors/summary_statistics/mean/mean/sblri-blr.json) file.

The reference posteriors for all elements of `beta` and `sigma` are all very close to 1.0.

The experiments reported in Figure 3 of the paper [Pathfinder: Parallel quasi-Newton variational inference](https://arxiv.org/abs/2108.03782) by Zhang et al. show that Pathfinder provides a better estimate of the posterior, as measured by the 1-Wasserstein distance to the reference posterior, than 75 iterations of the warmup Phase I algorithm used by the NUTS-HMC sampler.
Furthermore, Pathfinder is more computationally efficient, requiring fewer evaluations of the log density and gradient functions. Therefore, using the Pathfinder estimates to initialize the parameter values for the NUTS-HMC sampler can allow the sampler to do a better job of adapting the stepsize and metric during warmup, resulting in better performance and estimation.

We construct the parameter inits for full MCMC sampling below. The `.create_inits()` default behavior is to create inits for four chains to correspond with the sampling defaults. You can requests more or less by modifying the `chains` keyword argument.

In [3]:
pathfinder_inits = pathfinder_fit.create_inits()
for chain_init in pathfinder_inits:
    print(chain_init)

{'beta': array([1.000027  , 1.0003686 , 0.99944712, 1.0025016 , 1.0017672 ]), 'sigma': array(0.97164883)}
{'beta': array([0.999901  , 1.0020088 , 0.99929122, 1.0020285 , 0.99995542]), 'sigma': array(0.78456394)}
{'beta': array([0.99988369, 1.0008259 , 0.99928547, 1.0002089 , 1.001733  ]), 'sigma': array(0.93058113)}
{'beta': array([1.0010388, 1.0014929, 1.000171 , 1.0034019, 1.0011566]), 'sigma': array(1.1009083)}


We see that the Pathfinder inits are close the reference posteriors for the parameters. To use these inits, we pass the `pathfinder_inits` object to the `inits` kwarg:

In [4]:
mcmc_pathfinder_inits_fit = model.sample(
    data=data_file, inits=pathfinder_inits, iter_warmup=75, seed=12345
)

16:29:31 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

16:29:31 - cmdstanpy - INFO - CmdStan done processing.


16:29:31 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Consider re-running with show_console=True if the above output is unclear!


In [5]:
print(mcmc_pathfinder_inits_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  sigma
Such high values indicate incomplete mixing and biased estimation.
You should consider regularizing your model with additional prior information or a more effective parameterization.

Processing complete.



Despite only running 75 warmup iterations, all posterior diagnostics from the sampler look good.

In [6]:
mcmc_pathfinder_inits_fit.summary()

,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
lp__,-157.075000,0.096603,1.938160,1.747230,-160.646000,-156.695000,-154.65500,545.934,546.605,952.764,1.00658
beta[1],0.999475,0.000014,0.000984,0.000972,0.997847,0.999471,1.00109,5001.630,3233.990,8728.840,1.00107
beta[2],1.000280,0.000018,0.001186,0.001173,0.998303,1.000270,1.00222,4123.550,2723.710,7196.430,1.00146
beta[3],1.000430,0.000014,0.000955,0.000954,0.998842,1.000430,1.00200,4796.890,2802.500,8371.540,1.00042
beta[4],1.001140,0.000016,0.001082,0.001054,0.999406,1.001140,1.00294,4576.750,3113.860,7987.350,1.00137
beta[5],1.001530,0.000016,0.001036,0.001024,0.999835,1.001530,1.00323,4482.910,2741.530,7823.580,1.00107
sigma,0.974788,0.006300,0.081121,0.078815,0.859012,0.966043,1.11922,174.329,167.456,304.239,1.02759


If we were to instead use the default random parameter initializations, we would need to run more warmup iterations to produce useful samples. For example, if we only run the same 75 warmup iterations with random inits, the result fails to estimate `sigma` correctly: 

In [7]:
mcmc_random_inits_fit = model.sample(data=data_file, iter_warmup=75, seed=12345)

16:29:31 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

16:29:31 - cmdstanpy - INFO - CmdStan done processing.


16:29:31 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 10 divergent transitions (1.0%)
	Chain 3 had 174 divergent transitions (17.4%)
	Chain 4 had 105 divergent transitions (10.5%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [8]:
mcmc_random_inits_fit.summary()

,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
lp__,-186.905000,24.818700,35.341700,29.698100,-246.822000,-176.50200,-154.91900,4.98442,2000.000,20.8553,2.53979
beta[1],0.999441,0.000113,0.002364,0.001368,0.995958,0.99946,1.00339,484.04200,2000.000,2025.2800,1.15098
beta[2],1.000090,0.000134,0.002730,0.001847,0.995429,1.00018,1.00449,426.47500,2000.000,1784.4100,1.28431
beta[3],1.000420,0.000106,0.002270,0.001447,0.996795,1.00042,1.00404,501.70500,2000.000,2099.1800,1.16093
beta[4],1.001340,0.000144,0.002320,0.001551,0.997944,1.00132,1.00501,295.28900,216.765,1235.5200,1.14912
beta[5],1.001500,0.000143,0.002671,0.001631,0.997447,1.00139,1.00522,499.25100,2000.000,2088.9100,1.18556
sigma,1.860840,0.737960,1.053270,0.858795,0.892552,1.55046,3.63793,4.95282,2000.000,20.7231,2.72262


In [9]:
print(mcmc_random_inits_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
289 of 4000 (7.22%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
The E-BFMI, 0.01, is below the nominal threshold of 0.30 which suggests that HMC may have trouble exploring the target distribution.
If possible, try to reparameterize the model.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  beta[1], beta[2], beta[3], beta[4], beta[5], sigma
Such high values indicate incomplete mixing and biased estimation.
You should consider regularizing your model with additional prior information or 

The diagnostics clearly indicate problems with estimating `sigma`. In this case, it is necessary to run the model with at least 150 warmup iterations to produce a good set of estimates when starting from the default initialization.

### Other inference algorithms

We can follow the same pattern with Stan's ADVI algorithm by first using the `CmdStanModel.variationl` method. Because this algorithm is unstable and may fail to converge, we run it with argument `require_converged` set to `False`.  We also specify a seed, to avoid instabilities as well as for reproducibility.

In [10]:
vb_fit = model.variational(data=data_file, require_converged=False, seed=123)

16:29:31 - cmdstanpy - INFO - Chain [1] start processing


16:29:31 - cmdstanpy - INFO - Chain [1] done processing


16:29:31 - cmdstanpy - WARNING - The algorithm may not have converged.
Proceeding because require_converged is set to False


The ADVI algorithm provides estimates of all model parameters.

The `variational` method returns a `CmdStanVB` object, which similarly can construct a set of inits with the `.create_inits()` method:

In [11]:
vb_inits = vb_fit.create_inits()
for chain_init in vb_inits:
    print(chain_init)

{'beta': array([1.0062732 , 0.99851799, 1.0042977 , 0.99820144, 0.99172243]), 'sigma': array(1.6371057)}
{'beta': array([1.0096971 , 0.99721197, 1.0104545 , 0.99807629, 0.99284831]), 'sigma': array(1.4758395)}
{'beta': array([1.0034637 , 0.99863334, 1.0083972 , 0.99666249, 0.98951142]), 'sigma': array(1.5760709)}
{'beta': array([1.0051634 , 1.0044818 , 1.0124325 , 0.99488438, 0.98966442]), 'sigma': array(1.6946697)}


Which can be passed to the `inits` keyword argument of the sample method.

In [12]:
mcmc_vb_inits_fit = model.sample(
    data=data_file, inits=vb_inits, iter_warmup=75, seed=12345
)

16:29:32 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

16:29:32 - cmdstanpy - INFO - CmdStan done processing.


16:29:32 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Consider re-running with show_console=True if the above output is unclear!


In [13]:
print(mcmc_vb_inits_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  sigma
Such high values indicate incomplete mixing and biased estimation.
You should consider regularizing your model with additional prior information or a more effective parameterization.

Processing complete.



In [14]:
mcmc_vb_inits_fit.summary()

,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
lp__,-156.854000,0.060074,1.731770,1.592440,-160.116000,-156.523000,-154.65500,920.643,1600.090,1552.520,1.001840
beta[1],0.999463,0.000014,0.000965,0.000955,0.997883,0.999453,1.00106,4509.900,3335.430,7605.220,0.999963
beta[2],1.000220,0.000017,0.001138,0.001120,0.998353,1.000230,1.00208,4450.490,3116.740,7505.040,1.001410
beta[3],1.000440,0.000014,0.000928,0.000893,0.998904,1.000470,1.00195,4631.150,2971.430,7809.700,1.000850
beta[4],1.001140,0.000016,0.001062,0.001069,0.999366,1.001140,1.00289,4583.850,2570.760,7729.940,0.999843
beta[5],1.001560,0.000016,0.001068,0.001082,0.999788,1.001580,1.00334,4450.980,3083.900,7505.870,1.000250
sigma,0.970870,0.004393,0.069366,0.069170,0.863394,0.968880,1.09416,283.648,323.737,478.327,1.018080


The sampler estimates match the reference posterior with no diagnostic issues.

Inits can also be constructed from the `laplace` method:

In [15]:
laplace_inits = model.laplace_sample(data=data_file, seed=123).create_inits()

16:29:32 - cmdstanpy - INFO - Chain [1] start processing


16:29:32 - cmdstanpy - INFO - Chain [1] done processing


16:29:32 - cmdstanpy - INFO - Chain [1] start processing


16:29:32 - cmdstanpy - INFO - Chain [1] done processing


In [16]:
mcmc_laplace_inits_fit = model.sample(
    data=data_file, inits=laplace_inits, iter_warmup=75, seed=12345
)

16:29:32 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

16:29:32 - cmdstanpy - INFO - CmdStan done processing.


In [17]:
print(mcmc_laplace_inits_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete, no problems detected.



And the `optimize` method. Since optimizations attempts to return a posterior mode, this will initialize all chains at the same point, which is not typically ideal.

In [18]:
optimized_inits = model.optimize(data=data_file, seed=123).create_inits()

16:29:32 - cmdstanpy - INFO - Chain [1] start processing


16:29:32 - cmdstanpy - INFO - Chain [1] done processing


16:29:32 - cmdstanpy - WARNING - The default behavior of CmdStanMLE.stan_variable() will change in a future release to always return a numpy.ndarray, even for scalar variables.


In [19]:
mcmc_optimize_inits_fit = model.sample(
    data=data_file, inits=optimized_inits, iter_warmup=75, seed=12345
)

16:29:32 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

16:29:32 - cmdstanpy - INFO - CmdStan done processing.


In [20]:
print(mcmc_optimize_inits_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  sigma
Such high values indicate incomplete mixing and biased estimation.
You should consider regularizing your model with additional prior information or a more effective parameterization.

Processing complete.



It is also possible to use the output of the `sample()` method itself to construct inits to be fed into a future sampling run:

In [21]:
first_mcmc_inits = model.sample(
    data=data_file, iter_warmup=75, seed=12345
).create_inits()

16:29:32 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

16:29:32 - cmdstanpy - INFO - CmdStan done processing.


16:29:33 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 10 divergent transitions (1.0%)
	Chain 3 had 174 divergent transitions (17.4%)
	Chain 4 had 105 divergent transitions (10.5%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [22]:
second_mcmc_fit = model.sample(data=data_file, inits=first_mcmc_inits, iter_warmup=75, seed=12345)

16:29:33 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

16:29:33 - cmdstanpy - INFO - CmdStan done processing.


16:29:33 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Consider re-running with show_console=True if the above output is unclear!


In [23]:
print(second_mcmc_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  sigma
Such high values indicate incomplete mixing and biased estimation.
You should consider regularizing your model with additional prior information or a more effective parameterization.

Processing complete.



We see that despite the initial sampling issues in the first MCMC run, the inits sourced from that run result in reasonable sampling in the second run.